In [1]:
# using the datasets library from hugging face to handle downloading the dataset required for the task
from datasets import load_dataset

In [2]:
import torch
import torch.nn as nn 

In [3]:
imdb_dataset = load_dataset("imdb")
split = imdb_dataset['train'].train_test_split(train_size=0.8, seed=42)
imdb_train_set, imdb_valid_set = split['train'], split['test']
imdb_test_set = imdb_dataset['test']

Inspect a couple of reviews:

In [4]:
imdb_train_set[1]['text']

"'The Rookie' was a wonderful movie about the second chances life holds for us and also puts an emotional thought over the audience, making them realize that your dreams can come true. If you loved 'Remember the Titans', 'The Rookie' is the movie for you!! It's the feel good movie of the year and it is the perfect movie for all ages. 'The Rookie' hits a major home run!"

In [5]:
imdb_train_set[1]['label']

1

In [6]:
imdb_train_set[16]['text']

"Lillian Hellman's play, adapted by Dashiell Hammett with help from Hellman, becomes a curious project to come out of gritty Warner Bros. Paul Lukas, reprising his Broadway role and winning the Best Actor Oscar, plays an anti-Nazi German underground leader fighting the Fascists, dragging his American wife and three children all over Europe before finding refuge in the States (via the Mexico border). They settle in Washington with the wife's wealthy mother and brother, though a boarder residing in the manor is immediately suspicious of the newcomers and spends an awful lot of time down at the German Embassy playing poker. It seems to take forever for this drama to find its focus, and when we realize what the heart of the material is (the wise, honest, direct refugees teaching the clueless, head-in-the-sand Americans how the world has suddenly changed), it seems a little patronizing--the viewer is quite literally put in the relatives' place, being lectured to. Lukas has several speeches 

In [7]:
imdb_train_set[16]['label']

0

A simple char-rnn model would struggle; we need a more powerful tokenization technique, so lets focus on tokenization before turning on to sentiment analysis.

**Tokenization using the hugging face tokenizers library:**
- 2016 => several methods to tokenize and detokenize text at subword level explored. Sub-word levbel, why? this way even if your model encounters a rare word that its never seen before, it can still reasonably guess what it means. for example, if the model never saw the word "smartest" during training, if it learned the word "smart" and it also learned the suffix "est", it can infer the meaning of "smartest". One of the techniques used for sub-level word tokenization and detokenization is byte pair encoding (BPE). BPE works by splitting the whole training set into individual characters, and then at each iteration, finding the most frequent pairs of adjacent tokens and adds it to the vocabulary. It repeats this process until the vocabulary reaches the desired size. 
- The hugging face tokenizers library includes highly efficient implementations of several tokenization algorithms, including BPE. 

train a BPE model on the IMDB dataset....

In [8]:
import tokenizers

In [9]:
imdb_train_set

Dataset({
    features: ['text', 'label'],
    num_rows: 20000
})

In [10]:
imdb_train_set.num_rows

20000

In [11]:
def get_reviews(data):
    for i in range(data.num_rows):
        yield data[i]['text'] 

Code below:
- create a BPE model, specifying an unknown token "<unk>" which will be used later if we try to tokenize some text containing tokens the model never saw during training : the unknown tokens will be replaced with the "<unk>" token
- we then create a tokenizer based on the BPE model
- We specify a pre_tokenizer - in this case we use the Whitespace preprocessor which splits the text at spaces (and drops the spaces), and also separates groups of letters and groups of nonletters. For example "Hello, world!!" gets split into ["Hello",",","world","!!"]. The BPE algo then gets run on these individual chunks which imporives training speed and token quality
- We then define a list of special tokens: a padding token "<pad>" that comes in handy when we create batches of text of different lengths, and the unknown token already discussed
- Create a BpeTrainer, specifying the maximum vocabulary size and  the list of special tokens. The trainer will add the special tokens at the beginning of the vocab so "<pad>" token 0 and "<unk>" token 1.
- Lastly we train the tokenizer on the data using our custom generator func that yields the data from the training set. 

In [12]:
bpe_model = tokenizers.models.BPE(unk_token="<unk>")
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
special_tokens = ["<pad>","<unk>"]
bpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000, special_tokens=special_tokens)
bpe_tokenizer.train_from_iterator(get_reviews(imdb_train_set), trainer=bpe_trainer)

In [13]:
bpe_tokenizer.normalizer

In [14]:
some_review = "what an awesome movie! 😊"
bpe_encoding = bpe_tokenizer.encode(some_review)
bpe_encoding

Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [15]:
bpe_encoding.tokens

['what', 'an', 'aw', 'es', 'ome', 'movie', '!', '<unk>']

In [16]:
bpe_encoding.ids

[369, 175, 395, 185, 262, 247, 4, 1]

In [17]:
some_review = "What an awesome MOVIE! 😊"
bpe_encoding = bpe_tokenizer.encode(some_review)
bpe_encoding

Encoding(num_tokens=12, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [18]:
bpe_encoding.tokens

['What', 'an', 'aw', 'es', 'ome', 'M', 'O', 'V', 'I', 'E', '!', '<unk>']

In [19]:
bpe_encoding.ids

[831, 175, 395, 185, 262, 48, 50, 57, 44, 40, 4, 1]

this tokenizer is quite terrible at this stage as it doesn't normalize the text before processing it. important to note as well that the tokenizer lib provides a get_vocab() method which returns a dictionary mapping each token to its ID. You can also use the token_to_id() method to map a single token, or conversely use id_to_token() method to go from ID to token. or use the decode() mtd to convert a list of token IDs to a string.

In [20]:
bpe_tokenizer.decode(bpe_encoding.ids)

'What an aw es ome M O V I E !'

can also encode batches...

In [21]:
bpe_tokenizer.encode_batch(imdb_train_set['text'][:3])

[Encoding(num_tokens=294, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=118, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=292, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

to create a single integer tensor containing the token ids of all 3 reviews, we first ought to ensure they all have the same number of tokens -> not usually the case. We can use the tokenizer lib for this -> by asking the tokenizer to pad the shorter reviews with the padding token id till they are as long as the longest review in the batch. Can also ask the tokenizer to truncate any sequence longer than some max length.

In [23]:
bpe_tokenizer.enable_padding(pad_id=0,pad_token="<pad>")
bpe_tokenizer.enable_truncation(max_length=500)

now we can encode batches and create single tensors of all token ids...

In [24]:
bpe_encodings = bpe_tokenizer.encode_batch(imdb_train_set['text'][:3])
bpe_batch_ids = torch.tensor([encoding.ids for encoding in bpe_encodings])
bpe_batch_ids

tensor([[653, 437, 211, 281,  87, 822, 191, 791, 282,  68, 276,  77, 189, 514,
         700,  17, 845, 191, 791, 766, 249, 445, 184, 317,  68, 235, 893, 974,
         228,  86, 669, 195, 264, 205, 193, 179, 176, 196, 437,  11,  54, 409,
         187,   5,  49, 394,  48, 348,   5, 560,  54, 227, 272,  92,  47,  88,
         291,  87,  10,  86, 486, 580, 238, 433,  86, 176, 235, 186, 352,  15,
         402, 177, 952, 950, 730, 207, 700, 979, 176,  83, 563, 362, 453, 174,
         193, 417, 302, 476, 202, 257, 180, 250,  17, 653, 273,  15, 190,  10,
          86,  81, 451, 182, 445, 181, 220, 369, 190, 177,  17, 234, 645, 665,
         574, 809,  48, 370,  68, 236,  38, 307,  72, 188, 976, 177, 970, 301,
         981,  72, 209, 177, 760, 197, 841, 305,  17, 234, 685,  81, 226, 207,
          86, 191, 413, 759, 334, 256, 672, 244, 612, 973, 184,  17,  38, 307,
          72, 177,  73, 305, 331, 207, 187, 837,  15, 188, 981,  72, 209, 802,
         246, 191, 243, 715, 202, 175, 553, 182, 410

In [25]:
bpe_batch_ids.shape

torch.Size([3, 294])

each encoding object also includes an attention_mask attribute containing a 1 for each nonpadding token and a 0 for each padding token. This can be used in models to ignore padded timesteps by multiplying a tensor with the attention mask. Or use the list of valid seq lengths - here's how to get both

In [26]:
attention_mask = torch.tensor([encoding.attention_mask for encoding in bpe_encodings])
attention_mask

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1],


In [27]:
lengths = attention_mask.sum(dim=-1)

In [28]:
lengths

tensor([294, 118, 292])

testing the above again with the word piece tokenization algorithm similar to BPE but its token merges tend to favour more merges resulting in more meaningful words and often results in shorter sequence lengths than bpe or bbpe.

In [30]:
wp_model = tokenizers.models.WordPiece(unk_token="<unk>")
wp_normalizer = tokenizers.normalizers.Sequence([tokenizers.normalizers.NFD(), tokenizers.normalizers.Lowercase(), tokenizers.normalizers.StripAccents()])
wp_tokenizer = tokenizers.Tokenizer(wp_model)
wp_tokenizer.normalizer = wp_normalizer
wp_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
special_tokens = ["<pad>","<unk>"]
wp_trainer = tokenizers.trainers.WordPieceTrainer(vocab_size=10000, special_tokens=special_tokens)
wp_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
wp_tokenizer.train_from_iterator(get_reviews(imdb_train_set), trainer=wp_trainer)

In [31]:
wp_encoding = wp_tokenizer.encode("what an awesome movie! 😊")

In [32]:
wp_encoding.tokens

['what', 'an', 'awesome', 'movie', '!', '<unk>']

In [34]:
wp_encodings = wp_tokenizer.encode_batch(imdb_train_set['text'][:3])

In [37]:
wp_encodings[1].tokens

["'",
 'the',
 'rook',
 '##ie',
 "'",
 'was',
 'a',
 'wonderful',
 'movie',
 'about',
 'the',
 'second',
 'chances',
 'life',
 'holds',
 'for',
 'us',
 'and',
 'also',
 'puts',
 'an',
 'emotional',
 'thought',
 'over',
 'the',
 'audience',
 ',',
 'making',
 'them',
 'realize',
 'that',
 'your',
 'dreams',
 'can',
 'come',
 'true',
 '.',
 'if',
 'you',
 'loved',
 "'",
 'remember',
 'the',
 'tit',
 '##ans',
 "',",
 "'",
 'the',
 'rook',
 '##ie',
 "'",
 'is',
 'the',
 'movie',
 'for',
 'you',
 '!!',
 'it',
 "'",
 's',
 'the',
 'feel',
 'good',
 'movie',
 'of',
 'the',
 'year',
 'and',
 'it',
 'is',
 'the',
 'perfect',
 'movie',
 'for',
 'all',
 'ages',
 '.',
 "'",
 'the',
 'rook',
 '##ie',
 "'",
 'hits',
 'a',
 'major',
 'home',
 'run',
 '!',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',
 '<pad>',


training your own tokenizer is useful in many situations; for example if dealing with domain-specific text such as medical, legal, or engineering documents full of jargon, or if the text is written in a low-resource language or dialect, or code written in a new programming language. In some other cases, u can reuse a pretrained tokenizer.

**Reusing Pretrained Tokenizers:**

In [39]:
# get the pretrained tokenizer used by the GPT-2 model (bbpe)
import transformers

In [ ]:
# encoded = gpt2_tokenizer(
#     texts,
#     padding=True,          # or "max_length"
#     truncation=True,
#     max_length=128,
#     return_tensors="pt"
# )

In [40]:
gpt2_tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")
gpt2_encoding = gpt2_tokenizer(imdb_train_set['text'][:3], truncation=True, max_length=500)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [41]:
gpt2_encoding

{'input_ids': [[29391, 35030, 1690, 423, 257, 1688, 8046, 13, 1119, 1690, 1282, 503, 2045, 588, 257, 2646, 4676, 373, 2391, 4624, 319, 262, 3800, 357, 16678, 355, 366, 24732, 10584, 11074, 35727, 18348, 316, 338, 4571, 7622, 262, 2646, 6776, 11, 543, 318, 2592, 2408, 1201, 262, 4286, 4438, 683, 645, 1103, 4427, 13, 7831, 11, 340, 338, 3621, 284, 804, 379, 329, 644, 340, 318, 13, 383, 16585, 1022, 3899, 327, 5718, 290, 12803, 29030, 303, 318, 2407, 10457, 13, 383, 17262, 286, 511, 2776, 389, 6452, 13, 327, 5718, 318, 9623, 355, 1464, 11, 290, 29030, 303, 3011, 530, 286, 465, 1178, 8395, 284, 1107, 719, 29847, 1671, 1220, 6927, 1671, 11037, 40, 22127, 326, 314, 1053, 1239, 1775, 314, 430, 32325, 338, 711, 11, 475, 314, 3285, 326, 9180, 4332, 261, 9659, 338, 16711, 318, 17074, 13, 383, 4226, 318, 8131, 47370, 11, 290, 7622, 345, 25260, 13, 366, 20148, 46670, 1, 318, 281, 36005, 17774, 2646, 11, 290, 318, 7151, 329, 3016, 477, 3296, 286, 3800, 290, 3159, 29847, 1671, 1220, 6927, 1671, 1103

In [42]:
gpt2_token_ids = gpt2_encoding['input_ids'][0][:10]
gpt2_token_ids

[29391, 35030, 1690, 423, 257, 1688, 8046, 13, 1119, 1690]

In [43]:
gpt2_tokenizer.decode(gpt2_token_ids)

'Stage adaptations often have a major fault. They often'

pretrained word piece tokenizer:
- reuse the tokenizer of any LLM pretrained using WordPiece, such as BERT 
- this tokenizer has a padding token unlike the gpt2 tokenizer
- shortest lengths will be padded to the lengths of the longest one - this allows us to specify return_tensors = "pt" to get a pytorch tensor instead of a python list of lists 

In [51]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
bert_encoding = bert_tokenizer(imdb_train_set['text'][:3], padding=True, truncation=True, max_length=500, return_tensors="pt")

In [52]:
bert_encoding["input_ids"]

tensor([[  101,  2754, 17241,  2411,  2031,  1037,  2350,  6346,  1012,  2027,
          2411,  2272,  2041,  2559,  2066,  1037,  2143,  4950,  2001,  3432,
          2872,  2006,  1996,  2754,  1006,  2107,  2004,  1000,  2305,  2388,
          1000,  1007,  1012, 11430, 11320, 11368,  1005,  1055,  3257,  7906,
          1996,  2143,  4142,  1010,  2029,  2003,  2926,  3697,  2144,  1996,
          3861,  3253,  2032,  2053,  2613,  4119,  1012,  2145,  1010,  2009,
          1005,  1055,  3835,  2000,  2298,  2012,  2005,  2054,  2009,  2003,
          1012,  1996,  6370,  2090,  2745, 19881,  1998,  5696, 20726,  2003,
          3243,  8235,  1012,  1996, 10949,  1997,  2037,  3276,  2024, 11341,
          1012, 19881,  2003, 10392,  2004,  2467,  1010,  1998, 20726,  4152,
          2028,  1997,  2010,  2261,  9592,  2000,  2428,  2552,  1012,  1026,
          7987,  1013,  1028,  1026,  7987,  1013,  1028,  1045, 18766,  2008,
          1045,  1005,  2310,  2196,  2464, 11209, 2

In [53]:
bert_encoding["attention_mask"]

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

**Building and training a sentiment analysis model:**
- given the state of our data we need to apply tokenization to it and to do that we can either use the map method of our dataset or write the logic in the collate function of the DataLoader...

In [54]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased") # checkpoint for case insensitive BERT model pretrained on english text

In [55]:
imdb_train_set[0]

{'text': 'Stage adaptations often have a major fault. They often come out looking like a film camera was simply placed on the stage (Such as "Night Mother"). Sidney Lumet\'s direction keeps the film alive, which is especially difficult since the picture offered him no real challenge. Still, it\'s nice to look at for what it is. The chemistry between Michael Caine and Christopher Reeve is quite brilliant. The dynamics of their relationship are surprising. Caine is fantastic as always, and Reeve gets one of his few chances to really act.<br /><br />I confess that I\'ve never seen Ira Levin\'s play, but I hear that Jay Presson Allen\'s adaptation is faithful. The script is incredibly convoluted, and keeps you guessing. "Deathtrap" is an enormously entertaining film, and is recommended for nearly all fans of stage and screen.<br /><br />7.4 out of 10',
 'label': 1}

In [80]:
def collate_fn(batch, tokenizer=bert_tokenizer):
    reviews = [review['text'] for review in batch]
    labels = [review['label'] for review in batch]
    encodings = tokenizer(reviews, padding=True, truncation=True, max_length=200, return_tensors="pt")
    labels = torch.tensor(labels, dtype=torch.float32)
    return encodings,labels

In [81]:
from torch.utils.data import DataLoader

In [82]:
batch_size = 256

imdb_train_loader = DataLoader(imdb_train_set, batch_size=batch_size, collate_fn=collate_fn, shuffle=True)
imdb_valid_loader = DataLoader(imdb_valid_set, batch_size=batch_size, collate_fn=collate_fn)
imdb_test_loader = DataLoader(imdb_test_set, batch_size=batch_size, collate_fn=collate_fn)

Create the model:

In [88]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

In [90]:
class SentimentAnalysisModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128, hidden_dim=64, pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1)
    
    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(), batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        output = self.output(hidden_states[-1])
        return nn.functional.sigmoid(output)

In [91]:
bert_tokenizer.vocab_size

30522

In [93]:
from utils.early_stopping import EarlyStopping

In [92]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")
model = SentimentAnalysisModel(vocab_size=bert_tokenizer.vocab_size).to(device)

In [95]:
criterion = nn.BCELoss(reduction='sum')
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
early_stopper = EarlyStopping(patience=50, checkpoint_path="sent_classifier.pt", restore_best_weights=True, verbose=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',patience=5,factor=0.9)

In [99]:
n_epochs = 1000

train_loss = [0]*n_epochs
val_loss = [0]*n_epochs

for epoch in range(n_epochs):
    # set the model to training mode
    model.train() 
    # iterate through the training data
    for x_batch, y_batch in imdb_train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.unsqueeze(1).to(device)
        optimizer.zero_grad()
        out = model(x_batch)
        # adding the l1 norm
        norm = sum(p.abs().sum() for p in model.parameters())
        loss = criterion(out, y_batch) + 1e-5*norm
        # backpropagation
        loss.backward()
        # update gradients
        optimizer.step()
        train_loss[epoch]+=loss.item()
    train_loss[epoch] /= len(imdb_train_loader.dataset)

    # evaluation on the val set
    model.eval()
    with torch.no_grad():
        for x_batch, y_batch in imdb_valid_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.unsqueeze(1).to(device)
            out = model(x_batch)
            loss = criterion(out, y_batch)
            val_loss[epoch] += loss.item()
        val_loss[epoch] /= len(imdb_valid_loader.dataset)

        scheduler.step(val_loss[epoch])
        print(f'Epoch: {epoch+1}| Train loss: {train_loss[epoch]:.4f}| Val loss: {val_loss[epoch]:.4f}')
        early_stopper(val_loss[epoch], model, optimizer, epoch)
        if early_stopper.should_stop:
            print(f"Stopping at epoch: {epoch+1}")
            break



Epoch: 1| Train loss: 0.8026| Val loss: 0.6517
Metric improved to 0.6517. Checkpoint saved at epoch 0
Epoch: 2| Train loss: 0.7099| Val loss: 0.5428
Metric improved to 0.5428. Checkpoint saved at epoch 1
Epoch: 3| Train loss: 0.6237| Val loss: 0.4986
Metric improved to 0.4986. Checkpoint saved at epoch 2
Epoch: 4| Train loss: 0.5299| Val loss: 0.4494
Metric improved to 0.4494. Checkpoint saved at epoch 3
Epoch: 5| Train loss: 0.4507| Val loss: 0.4283
Metric improved to 0.4283. Checkpoint saved at epoch 4
Epoch: 6| Train loss: 0.3962| Val loss: 0.4016
Metric improved to 0.4016. Checkpoint saved at epoch 5
Epoch: 7| Train loss: 0.3420| Val loss: 0.3806
Metric improved to 0.3806. Checkpoint saved at epoch 6
Epoch: 8| Train loss: 0.2899| Val loss: 0.4751
No improvement for 1 epoch(s)
Epoch: 9| Train loss: 0.2493| Val loss: 0.4333
No improvement for 2 epoch(s)
Epoch: 10| Train loss: 0.2178| Val loss: 0.4635
No improvement for 3 epoch(s)
Epoch: 11| Train loss: 0.1876| Val loss: 0.4843
No imp

KeyboardInterrupt: 

Had to interrupt above to save time..
- now reload from checkpoint and eval the model on the test set...

In [102]:
state_dict = torch.load('sent_classifier.pt')

In [103]:
state_dict

{'epoch': 6,
 'model_state_dict': OrderedDict([('embed.weight',
               tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
                         0.0000e+00,  0.0000e+00],
                       [-8.2797e-05, -1.1398e-04,  5.7724e-01,  ...,  1.0075e-01,
                         1.2799e+00,  4.3055e-01],
                       [-7.3614e-05,  1.7966e-04,  7.7394e-01,  ...,  1.6169e-01,
                        -9.8497e-01,  3.6191e-01],
                       ...,
                       [ 6.9650e-05,  9.3649e-05, -5.8052e-06,  ..., -1.2127e-01,
                         3.4658e-02, -4.5287e-05],
                       [ 4.2022e-05, -8.6932e-01,  3.9856e-01,  ..., -3.3700e-01,
                        -4.4412e-04, -1.8631e-04],
                       [ 6.9457e-06, -3.2981e-04, -2.1382e-04,  ..., -7.6137e-01,
                        -4.7560e-01,  1.6471e-06]], device='mps:0')),
              ('gru.weight_ih_l0',
               tensor([[-0.0105, -0.1417,  0.1256,  ...,

In [104]:
trained_model = SentimentAnalysisModel(vocab_size=bert_tokenizer.vocab_size)
trained_model.load_state_dict(state_dict['model_state_dict'])
trained_model.to(device)

SentimentAnalysisModel(
  (embed): Embedding(30522, 128, padding_idx=0)
  (gru): GRU(128, 64, num_layers=2, batch_first=True, dropout=0.2)
  (output): Linear(in_features=64, out_features=1, bias=True)
)

Evaluate on the test set:

In [108]:
trained_model.eval() # set the model to evaluation mode


test_acc = 0
# turn off gradient tracking
with torch.no_grad():
    for x_batch, y_batch in imdb_test_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.unsqueeze(1).to(device)
        pred = trained_model(x_batch)
        pred = pred >= 0.5 # broadcast and return booleans
        # compare with the targets
        pred_comp = (pred == y_batch).sum().item()
        test_acc += pred_comp
    test_acc = test_acc/len(imdb_test_loader.dataset)


In [109]:
test_acc

0.826

testing the encodings..

In [110]:
imdb_train_set

Dataset({
    features: ['text', 'label'],
    num_rows: 20000
})

In [111]:
z = imdb_train_set['text'][:100]

z_encodings = bert_tokenizer(z, padding=True, truncation=True, max_length=200, return_tensors="pt")

In [113]:
z_encodings['input_ids'].shape

torch.Size([100, 200])

In [115]:
z_encodings['attention_mask'].shape

torch.Size([100, 200])

In [116]:
z_encodings['attention_mask'].sum(dim=1).shape

torch.Size([100])

In [ ]:
out = torch.cat((hidden[-2,:,:], hidden[-1, :, :]), dim=1)

In [117]:
a = torch.rand(256, 64)
b = torch.rand(256, 64)  

c = torch.cat((a,b),dim=1)

In [118]:
c.shape

torch.Size([256, 128])

Bidirectional model:

In [122]:
class BiSentimentAnalysisModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128, hidden_dim=64, pad_id=0,dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,batch_first=True, bidirectional=True, dropout=dropout)
        self.output = nn.Linear(2*hidden_dim,1)
    
    def forward(self, encodings):
        embeddings = self.embed(encodings['input_ids'])
        lengths = encodings['attention_mask'].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(), batch_first=True, enforce_sorted=False)
        _outputs,hidden_states = self.gru(packed)
        out = torch.cat((hidden_states[-2,:,:],hidden_states[-1,:,:]), dim=1)
        out = self.output(out)
        return nn.functional.sigmoid(out)

In [123]:
bert_tokenizer.vocab_size

30522

In [125]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")
model = BiSentimentAnalysisModel(vocab_size=bert_tokenizer.vocab_size).to(device)
criterion = nn.BCELoss(reduction='sum')
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
early_stopper = EarlyStopping(patience=10,checkpoint_path='sent_classifier2.pt', restore_best_weights=True, verbose=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.9)

In [126]:
n_epochs = 50

train_loss = [0]*n_epochs
val_loss = [0]*n_epochs

for epoch in range(n_epochs):
    # set the model to training mode
    model.train() 
    # iterate through the training data
    for x_batch, y_batch in imdb_train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.unsqueeze(1).to(device)
        optimizer.zero_grad()
        out = model(x_batch)
        # adding the l1 norm
        norm = sum(p.abs().sum() for p in model.parameters())
        loss = criterion(out, y_batch) + 1e-5*norm
        # backpropagation
        loss.backward()
        # update gradients
        optimizer.step()
        train_loss[epoch]+=loss.item()
    train_loss[epoch] /= len(imdb_train_loader.dataset)

    # evaluation on the val set
    model.eval()
    with torch.no_grad():
        for x_batch, y_batch in imdb_valid_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.unsqueeze(1).to(device)
            out = model(x_batch)
            loss = criterion(out, y_batch)
            val_loss[epoch] += loss.item()
        val_loss[epoch] /= len(imdb_valid_loader.dataset)

        scheduler.step(val_loss[epoch])
        print(f'Epoch: {epoch+1}| Train loss: {train_loss[epoch]:.4f}| Val loss: {val_loss[epoch]:.4f}')
        early_stopper(val_loss[epoch], model, optimizer, epoch)
        if early_stopper.should_stop:
            print(f"Stopping at epoch: {epoch+1}")
            break

Epoch: 1| Train loss: 0.7972| Val loss: 0.6320
Metric improved to 0.6320. Checkpoint saved at epoch 0
Epoch: 2| Train loss: 0.6750| Val loss: 0.6528
No improvement for 1 epoch(s)
Epoch: 3| Train loss: 0.6030| Val loss: 0.5169
Metric improved to 0.5169. Checkpoint saved at epoch 2
Epoch: 4| Train loss: 0.4891| Val loss: 0.4118
Metric improved to 0.4118. Checkpoint saved at epoch 3
Epoch: 5| Train loss: 0.4179| Val loss: 0.4278
No improvement for 1 epoch(s)
Epoch: 6| Train loss: 0.3592| Val loss: 0.4112
Metric improved to 0.4112. Checkpoint saved at epoch 5
Epoch: 7| Train loss: 0.3033| Val loss: 0.3883
Metric improved to 0.3883. Checkpoint saved at epoch 6
Epoch: 8| Train loss: 0.2539| Val loss: 0.4139
No improvement for 1 epoch(s)


KeyboardInterrupt: 

**Reusing pretrained embeddings and language models:**
- we can reuse word embeddings even when they were trained on some other corpus, even if it wasn't composed of movie reviews, and even if they were not trained for sentiment analysis. 
- Since we used pretrained tokens for the BERT model, might as well try using its embedding layer. First, we need to down;oad the pretrained model using the AutoModel.from_pretrained() function of the transformers library. 

In [127]:
bert_model = transformers.AutoModel.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [128]:
# directly accessing the model's embeddings...
bert_model.embeddings.word_embeddings

Embedding(30522, 768, padding_idx=0)

In [131]:
bert_model.embeddings.word_embeddings.weight.data

tensor([[-0.0102, -0.0615, -0.0265,  ..., -0.0199, -0.0372, -0.0098],
        [-0.0117, -0.0600, -0.0323,  ..., -0.0168, -0.0401, -0.0107],
        [-0.0198, -0.0627, -0.0326,  ..., -0.0165, -0.0420, -0.0032],
        ...,
        [-0.0218, -0.0556, -0.0135,  ..., -0.0043, -0.0151, -0.0249],
        [-0.0462, -0.0565, -0.0019,  ...,  0.0157, -0.0139, -0.0095],
        [ 0.0015, -0.0821, -0.0160,  ..., -0.0081, -0.0475,  0.0753]])

In [132]:
bert_model.embeddings.word_embeddings.weight.data.shape

torch.Size([30522, 768])

In [134]:
class SentimentAnalysisPreEmbeds(nn.Module):
    def __init__(self, pretrained_embeddings, n_layers=2, hidden_dim=64, dropout=0.2):
        super().__init__()
        weights = pretrained_embeddings.weight.data
        self.embed = nn.Embedding.from_pretrained(weights, freeze=True)
        embed_dim = weights.shape[-1]
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,batch_first=True,dropout=dropout,bidirectional=True)
        self.output = nn.Linear(2*hidden_dim,1)
    
    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        out = torch.cat((hidden_states[-2,:,:], hidden_states[-1,:,:]), dim=1) # concat the 2 bidirectional output sequences
        out = self.output(out)
        return nn.functional.sigmoid(out)

In [137]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")
model = SentimentAnalysisPreEmbeds(pretrained_embeddings=bert_model.embeddings.word_embeddings).to(device)
criterion = nn.BCELoss(reduction='sum')
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
early_stopper = EarlyStopping(patience=10,checkpoint_path='sent_classifier3.pt', restore_best_weights=True, verbose=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.9)

In [138]:
n_epochs = 50

train_loss = [0]*n_epochs
val_loss = [0]*n_epochs

for epoch in range(n_epochs):
    # set the model to training mode
    model.train() 
    # iterate through the training data
    for x_batch, y_batch in imdb_train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.unsqueeze(1).to(device)
        optimizer.zero_grad()
        out = model(x_batch)
        # adding the l1 norm
        norm = sum(p.abs().sum() for p in model.parameters())
        loss = criterion(out, y_batch) + 1e-5*norm
        # backpropagation
        loss.backward()
        # update gradients
        optimizer.step()
        train_loss[epoch]+=loss.item()
    train_loss[epoch] /= len(imdb_train_loader.dataset)

    # evaluation on the val set
    model.eval()
    with torch.no_grad():
        for x_batch, y_batch in imdb_valid_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.unsqueeze(1).to(device)
            out = model(x_batch)
            loss = criterion(out, y_batch)
            val_loss[epoch] += loss.item()
        val_loss[epoch] /= len(imdb_valid_loader.dataset)

        scheduler.step(val_loss[epoch])
        print(f'Epoch: {epoch+1}| Train loss: {train_loss[epoch]:.4f}| Val loss: {val_loss[epoch]:.4f}')
        early_stopper(val_loss[epoch], model, optimizer, epoch)
        if early_stopper.should_stop:
            print(f"Stopping at epoch: {epoch+1}")
            break

Epoch: 1| Train loss: 0.7100| Val loss: 0.6779
Metric improved to 0.6779. Checkpoint saved at epoch 0
Epoch: 2| Train loss: 0.5639| Val loss: 0.4675
Metric improved to 0.4675. Checkpoint saved at epoch 1
Epoch: 3| Train loss: 0.5019| Val loss: 0.5359
No improvement for 1 epoch(s)
Epoch: 4| Train loss: 0.4824| Val loss: 0.4647
Metric improved to 0.4647. Checkpoint saved at epoch 3
Epoch: 5| Train loss: 0.4739| Val loss: 0.4140
Metric improved to 0.4140. Checkpoint saved at epoch 4
Epoch: 6| Train loss: 0.4431| Val loss: 0.4245
No improvement for 1 epoch(s)
Epoch: 7| Train loss: 0.4248| Val loss: 0.5009
No improvement for 2 epoch(s)
Epoch: 8| Train loss: 0.4276| Val loss: 0.3718
Metric improved to 0.3718. Checkpoint saved at epoch 7
Epoch: 9| Train loss: 0.4000| Val loss: 0.4116
No improvement for 1 epoch(s)
Epoch: 10| Train loss: 0.4105| Val loss: 0.3603
Metric improved to 0.3603. Checkpoint saved at epoch 9
Epoch: 11| Train loss: 0.3904| Val loss: 0.3821
No improvement for 1 epoch(s)
E

KeyboardInterrupt: 

Interrupting to save time..
- eval on test set with bert's non-contextualised embeddings...

In [139]:
state_dict = torch.load('sent_classifier3.pt')
trained_model = SentimentAnalysisPreEmbeds(pretrained_embeddings=bert_model.embeddings.word_embeddings)
trained_model.load_state_dict(state_dict['model_state_dict'])
trained_model.to(device)

SentimentAnalysisPreEmbeds(
  (embed): Embedding(30522, 768)
  (gru): GRU(768, 64, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
  (output): Linear(in_features=128, out_features=1, bias=True)
)

In [140]:
trained_model.eval() # set the model to evaluation mode


test_acc = 0
# turn off gradient tracking
with torch.no_grad():
    for x_batch, y_batch in imdb_test_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.unsqueeze(1).to(device)
        pred = trained_model(x_batch)
        pred = pred >= 0.5 # broadcast and return booleans
        # compare with the targets
        pred_comp = (pred == y_batch).sum().item()
        test_acc += pred_comp
    test_acc = test_acc/len(imdb_test_loader.dataset)

In [141]:
test_acc

0.8384

Pretrained word embeddings have been popular for quite a while. However, this approach has its limits. In particular, a word has a single representation, no matter the context. For example, the wors 'right' is encoded the same way in 'left and right' and 'right and wrong' even if it means different things. To address this limitation, Embeddings from Language Models (ELMo) introduced: these are contextualised word embeddings learned from the internal states of a deep bidirectional rnn language model. In other words, instead of just using pretrained word embeddings in your model, you can reuse several layers of a pretrained language model. 
- elmo predates transformers and BERT.
- elmo produced contextualised embeddings using deep bidirectional lstms, not attention. 

We now reuse the entire pretrained BERT model for our sent analysis task - extracting the contextualised embeddings..

In [142]:
bert_encoding = bert_tokenizer(imdb_train_set['text'][:3], padding=True, max_length=200, truncation=True,
                               return_tensors='pt')

In [144]:
bert_output = bert_model(**bert_encoding)

In [145]:
bert_output

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.2470, -0.4977,  0.1271,  ...,  0.0853,  0.5811,  0.4216],
         [ 0.0992,  0.4679, -0.2322,  ..., -0.0807,  0.8464, -0.6777],
         [ 0.9692,  0.2901, -0.1934,  ..., -0.1843,  0.0797,  0.0119],
         ...,
         [ 0.0884,  0.3943, -0.0035,  ...,  0.4302,  0.3150, -0.1580],
         [-0.0762, -0.2808,  0.8119,  ...,  0.4624,  0.8381, -0.3956],
         [ 0.2624, -0.2101,  0.6697,  ...,  0.4399,  0.6036, -0.3578]],

        [[-0.0101, -0.3241,  0.3658,  ..., -0.4497,  0.5121, -0.1260],
         [-0.2964, -0.2872, -0.5425,  ...,  0.6877,  1.3121, -1.5069],
         [-0.4449, -0.8080, -0.4515,  ...,  0.5536,  0.8961, -0.3652],
         ...,
         [ 0.3836, -0.1907,  0.3335,  ..., -0.0445, -0.1109,  0.0854],
         [ 0.3802,  0.0156,  0.4263,  ...,  0.1164,  0.1199, -0.2163],
         [ 0.1742, -0.0786,  0.5463,  ..., -0.0133, -0.2344, -0.1866]],

        [[ 0.0737, -0.7819,  0.1337,  ...,  0.2590,  

bert's output includes an attribute named last_hidden_state which contains contextualised embeddings for each token. The word last in this case referes to the last layer, not the last time step (BERT is a transformer not an RNN). This last_hidden_state tensor has a shape of [batch_size, max_sequence length, hidden size]. 

In [152]:
bert_output.keys()

odict_keys(['last_hidden_state', 'pooler_output'])

In [154]:
bert_model.config.hidden_size

768

In [155]:
bert_model.config

BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "dtype": "float32",
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.56.2",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

In [153]:
bert_output.last_hidden_state.shape

torch.Size([3, 200, 768])

In [174]:
bert_output.pooler_output.shape

torch.Size([3, 768])

using these contextualised embeddings in the sentiment analysis model...

In [164]:
batch_size = 32

imdb_train_loader = DataLoader(imdb_train_set, batch_size=batch_size, collate_fn=collate_fn, shuffle=True)
imdb_valid_loader = DataLoader(imdb_valid_set, batch_size=batch_size, collate_fn=collate_fn)
imdb_test_loader = DataLoader(imdb_test_set, batch_size=batch_size, collate_fn=collate_fn)

In [165]:
class SentimentAnalysisModelBert(nn.Module):
    def __init__(self, n_layers=2, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        # don't want to fine tune bert now so will freeze it
        # for param in self.bert.parameters():
        #     param.requires_grad = False
        # self.bert.eval() # setting bert to eval mode
        embed_dim = self.bert.config.hidden_size
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1)
    
    def forward(self, encodings):
        # with torch.no_grad():
        contextualized_embeddings = self.bert(**encodings).last_hidden_state
        lengths = encodings['attention_mask'].sum(dim=1)
        packed = pack_padded_sequence(contextualized_embeddings, lengths=lengths.cpu(), batch_first=True, enforce_sorted=False)
        _outputs,hidden_states = self.gru(packed)
        return nn.functional.sigmoid(self.output(hidden_states[-1,:,:]))

In [166]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")
model = SentimentAnalysisModelBert().to(device)
model.bert.requires_grad_(False)
criterion = nn.BCELoss(reduction='sum')
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
early_stopper = EarlyStopping(patience=10,checkpoint_path='sent_classifier3.pt', restore_best_weights=True, verbose=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.9)

In [167]:
n_epochs = 50

train_loss = [0]*n_epochs
val_loss = [0]*n_epochs

for epoch in range(n_epochs):
    # set the model to training mode
    model.train() 
    # iterate through the training data
    for x_batch, y_batch in imdb_train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.unsqueeze(1).to(device)
        optimizer.zero_grad()
        out = model(x_batch)
        # adding the l1 norm
        norm = sum(p.abs().sum() for p in model.parameters())
        loss = criterion(out, y_batch) + 1e-5*norm
        # backpropagation
        loss.backward()
        # update gradients
        optimizer.step()
        train_loss[epoch]+=loss.item()
    train_loss[epoch] /= len(imdb_train_loader.dataset)

    # evaluation on the val set
    model.eval()
    with torch.no_grad():
        for x_batch, y_batch in imdb_valid_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.unsqueeze(1).to(device)
            out = model(x_batch)
            loss = criterion(out, y_batch)
            val_loss[epoch] += loss.item()
        val_loss[epoch] /= len(imdb_valid_loader.dataset)

        scheduler.step(val_loss[epoch])
        print(f'Epoch: {epoch+1}| Train loss: {train_loss[epoch]:.4f}| Val loss: {val_loss[epoch]:.4f}')
        early_stopper(val_loss[epoch], model, optimizer, epoch)
        if early_stopper.should_stop:
            print(f"Stopping at epoch: {epoch+1}")
            break

Epoch: 1| Train loss: 1.4941| Val loss: 0.2707
Metric improved to 0.2707. Checkpoint saved at epoch 0
Epoch: 2| Train loss: 1.3935| Val loss: 0.2701
Metric improved to 0.2701. Checkpoint saved at epoch 1


KeyboardInterrupt: 

In [169]:
state_dict = torch.load('sent_classifier3.pt')
trained_model = SentimentAnalysisModelBert()
trained_model.load_state_dict(state_dict['model_state_dict'])
trained_model.to(device)

SentimentAnalysisModelBert(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [170]:
trained_model.eval() # set the model to evaluation mode


test_acc = 0
# turn off gradient tracking
with torch.no_grad():
    for x_batch, y_batch in imdb_test_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.unsqueeze(1).to(device)
        pred = trained_model(x_batch)
        pred = pred >= 0.5 # broadcast and return booleans
        # compare with the targets
        pred_comp = (pred == y_batch).sum().item()
        test_acc += pred_comp
    test_acc = test_acc/len(imdb_test_loader.dataset)

In [171]:
test_acc

0.89128

89% test acc with just 2 epochs - using contextualised embeddings..

could have frozen as well by using: model.bert.requires_grad_(False).
- another option for using contextualised embeddings is to use only the contextualised embedding for the very first token, which is the class token [CLS]. Indeed, during pretraining, the BERT model had to perform a text classification task based solely on this token's contextualised embedding. As a result, the model learned to summarise the most important features of the text into this embedding (embedding single embedding linking to the cls token). This simplifies the model quite a bit since we can now get rid of the nn.GRU module altogether, and the forward() method then becomes much shorter..
- cana get rid of the GRU as this cls embedding now is equivalent to the last hidden state output we are extracting from the gru

In [176]:
class SentimentAnalysisModelBert2(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output = nn.Linear(self.bert.config.hidden_size, 1)
    
    def forward(self, encodings):
        bert_output = self.bert(**encodings)
        return nn.functional.sigmoid(self.output(bert_output.last_hidden_state[:,0]))

in fact, the bert model contains an extra hidden layer on top of the class embedding, composed of an nn.Linear module and an nn.tanh module. This hidden layer is known as the pooler. to use it, replace bert_output.last_hidden_state[:,0] with bert_output.pooler_output/ may want to unfreeze the pooler after a few epochs to fine-tine it for the imdb task.
- started by reusing only a pretrained tokenizer, then reused the pretrained embeddings, then most of the pretrained bert model (contextualised embeddings), and finally the full model adding only an nn.linear layer ontop of the pooler layer.

In [177]:
# using the pooler output of the bert model:
class SentimentAnalysisModelBert3(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output = nn.Linear(self.bert.config.hidden_size, 1)
    def forward(self, encodings):
        bert_output = self.bert(**encodings)
        return nn.functional.sigmoid(self.output(bert_output.pooler_output))

Training time....

In [ ]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")
model = SentimentAnalysisModelBert2().to(device)
model.bert.requires_grad_(False)
criterion = nn.BCELoss(reduction='sum')
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
early_stopper = EarlyStopping(patience=10,checkpoint_path='sent_classifier3.pt', restore_best_weights=True, verbose=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=1, factor=0.9)

In [179]:
n_epochs = 5

train_loss = [0]*n_epochs
val_loss = [0]*n_epochs

for epoch in range(n_epochs):
    # set the model to training mode
    model.train() 
    # iterate through the training data
    for x_batch, y_batch in imdb_train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.unsqueeze(1).to(device)
        optimizer.zero_grad()
        out = model(x_batch)
        # adding the l1 norm
        norm = sum(p.abs().sum() for p in model.parameters())
        loss = criterion(out, y_batch) + 1e-5*norm
        # backpropagation
        loss.backward()
        # update gradients
        optimizer.step()
        train_loss[epoch]+=loss.item()
    train_loss[epoch] /= len(imdb_train_loader.dataset)

    # evaluation on the val set
    model.eval()
    with torch.no_grad():
        for x_batch, y_batch in imdb_valid_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.unsqueeze(1).to(device)
            out = model(x_batch)
            loss = criterion(out, y_batch)
            val_loss[epoch] += loss.item()
        val_loss[epoch] /= len(imdb_valid_loader.dataset)

        scheduler.step(val_loss[epoch])
        print(f'Epoch: {epoch+1}| Train loss: {train_loss[epoch]:.4f}| Val loss: {val_loss[epoch]:.4f}')
        early_stopper(val_loss[epoch], model, optimizer, epoch)
        if early_stopper.should_stop:
            print(f"Stopping at epoch: {epoch+1}")
            break

KeyboardInterrupt: 

we now can go a step further and use an off the shelf class for sentence classification.....